<a id='Top'></a>

# Prepare omic data <a class='tocSkip'></a>

Cancer is fundamentally a genome disease, and these genetic abnormalities can affect various molecular levels, including DNA sequences, gene expression, epigenetic modifications (such as miRNAs, non-coding RNAs, and methylation), and protein expression or structure. Each alteration carries distinct functional consequences that can influence tumor behavior, progression, and treatment response.

Understanding and integrating these molecular layers — often referred to as *omics* data — is essential for uncovering the biological mechanisms underlying cancer and for identifying potential therapeutic targets and biomarkers. The preprocessing of omic data is a crucial first step in this process. It ensures data quality, consistency, and compatibility across modalities and samples, enabling robust downstream analysis and modeling.

This notebook focuses on the preprocessing of multi-omic datasets, including but not limited to:
- **DNA Methylation (dnam)** 
- **Copy Number Variation (CNV)**
- **Micro RNA (miRNA)** 
- **Gene expression (RNA-seq)** 

The following figure illustrates the preprocessing workflow applied to each omic modality within the dataset.

<p align="left">
  <img src="../data_download/Fig/omic_preprocessing.png" alt="Omic Preprocessing" width="750"/>
</p>



# DNA Methylation

The data are provided in tables of array results of the level of methylation at known CpG sites. They include unique ids for the array probes and methylation Beta values, representing the ratio between the methylated array intensity and total array intensity (falls between 0, lower levels of methylation, and 1, higher levels of methylation). As explained in the GDC [Methylation Liftover Pipeline page](https://docs.gdc.cancer.gov/Data/Bioinformatics_Pipelines/Methylation_LO_Pipeline/), data was generated using either Illumina Infinium Human Methylation 27 (HM27; 27'578 probes) or HumanMethylation450 (HM450; 485'577 probes).

We use the intersection of the probes between the two (25'978 probes).

In [1]:
# Import necessary libraries
import os
import pandas as pd   
import numpy as np
from pathlib import Path

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.decomposition import PCA 
import warnings

In [2]:
# Define base and output directories
base_path = "../data"
output_path = "../data_download/pkl"
output_pickle_file_template = os.path.join(output_path, 'adn_{}.pkl')

# Define manifest file paths for BRCA and LGG
manifest_paths = {
    'BRCA': '../data_download/Manifest/BRCA/filtered_manifest/filtered_gdc_manifest.adn.txt',
    'LGG': '../data_download/Manifest/LGG/filtered_manifest/filtered_gdc_manifest.adn.txt',
}

# Iterate over each cancer type
for cancer_type in ['BRCA', 'LGG']:
    cancer_path = os.path.join(base_path, cancer_type)
    folder_path = os.path.join(cancer_path, 'omics', 'adn')

    if not os.path.isdir(folder_path):
        print(f"Directory not found: {folder_path}")
        continue

    # Load manifest to preserve the patient order
    manifest_df = pd.read_csv(manifest_paths[cancer_type], sep='\t')
    manifest_ids = manifest_df['id'].tolist()

    data_frames = []  # List to hold patient Beta_value columns
    composite_ref = None  # Will store the common 'Composite Element REF'

    # Collect each patient’s data
    for patient_id in manifest_ids:
        patient_path = os.path.join(folder_path, patient_id)

        if not os.path.isdir(patient_path):
            data_frames.append(pd.Series([None], name=patient_id))  # Placeholder if folder is missing
            continue

        # Look for the methylation data file
        methylation_file = next(
            (f for f in os.listdir(patient_path) if f.endswith('methylation_array.sesame.level3betas.txt')),
            None
        )

        if methylation_file:
            file_path = os.path.join(patient_path, methylation_file)
            df = pd.read_csv(file_path, sep='\t', header=None)
            df.columns = ['Composite Element REF', 'Beta_value']

            if composite_ref is None:
                composite_ref = df[['Composite Element REF']].reset_index(drop=True)

            beta_series = df['Beta_value'].reset_index(drop=True)
        else:
            beta_series = pd.Series([None] * len(composite_ref) if composite_ref is not None else [None])

        beta_series.name = patient_id
        data_frames.append(beta_series)

    # Validate collected data
    if composite_ref is None or not any(isinstance(col, pd.Series) for col in data_frames):
        print(f"No valid data found for {cancer_type}.")
        continue

    # Combine all patient data in one DataFrame efficiently
    patient_data_df = pd.concat(data_frames, axis=1)

    # Combine with Composite Element REF
    final_df = pd.concat([composite_ref, patient_data_df], axis=1)

    # Save to pickle file
    os.makedirs(output_path, exist_ok=True)
    output_pickle_file = output_pickle_file_template.format(cancer_type)
    final_df.to_pickle(output_pickle_file)
    print(f"Saved combined data for {cancer_type} to {output_pickle_file}")


Saved combined data for BRCA to ../data_download/pkl/adn_BRCA.pkl
Saved combined data for LGG to ../data_download/pkl/adn_LGG.pkl


In [3]:
# Base path where the .pkl files are stored
output_path = "../data_download/pkl"
cancer_types = ['LGG', 'BRCA']

# Iterate through each cancer type and check the dimensions of the .pkl files
for cancer_type in cancer_types:
    pickle_file_path = os.path.join(output_path, f'adn_{cancer_type}.pkl')

    if os.path.exists(pickle_file_path):   
        combined_df = pd.read_pickle(pickle_file_path)

        print(f"\nDisplaying the DataFrame for {cancer_type}:")
        display(combined_df)  # Display the full DataFrame

        # Show the dimensions of the DataFrame
        print("\nDimensions of the DataFrame:")
        print(combined_df.shape)
    else:
        print(f"\nThe file {pickle_file_path} does not exist.")


Displaying the DataFrame for LGG:


,Composite Element REF,6c402b0f-52f3-427c-bb3c-6e697062b3bc,db8006b0-3eeb-4408-bc61-357015169816,70aade5f-187e-4500-87df-f9996424de7d,0e528a4b-b052-4eeb-81df-67bbb7081300,20322683-b95c-4e44-aff4-3b25114c9635,baa5cded-bcce-4355-8179-faa164273669,a754cf15-3da9-4eb8-acf5-5e3aafdf64ee,dc4a02bc-9638-490c-afe2-d019b7a20feb,48e5fb3f-e0e9-4a4a-a63c-b2172ab87890,...,c9a39398-969e-402b-bf24-6f3f34d6b5fd,6b980248-f88e-4749-aa14-d8f49775f35e,9e2585f8-b0bf-48ec-b692-8d519e7684ef,75185c70-7ed1-4a8d-aeae-d6cebc80df74,d94632dd-7d8e-4e49-a62e-be83639a2dd4,650a115c-d1a9-4416-a15a-c8fd5d1d50ec,e6dd298e-a774-4074-bb3a-3b54e6564e18,9e9d9baf-0f64-4394-ae80-f94d1d413260,346c454b-6bbb-492a-b81d-926b5952f63a,43ee947f-d88f-434d-8780-6656fec1991e
0,cg00000029,0.690103,0.657379,0.831489,0.742158,0.639514,0.822372,0.897817,0.846715,0.661744,...,0.867569,0.579278,0.880285,0.516879,0.531778,0.682903,0.351138,0.889694,NaN,0.871403
1,cg00000108,0.957889,0.967432,0.956029,0.949274,0.956379,0.974914,0.968238,0.966660,0.962665,...,0.958231,0.975958,0.964912,0.954173,0.964141,0.972822,0.967945,0.956179,0.944520,0.966934
2,cg00000109,0.803878,0.916410,0.869470,0.911649,0.854345,0.909155,0.934060,0.883908,0.881226,...,0.816603,0.411459,0.932143,0.767400,0.913774,0.893947,0.904613,0.905384,0.842569,0.894141
3,cg00000165,0.615174,0.099334,0.199279,0.911272,0.254918,0.105330,0.152506,0.437856,0.370842,...,NaN,0.151610,0.121919,0.611562,0.154643,0.838444,0.508291,NaN,0.716442,0.582031
4,cg00000236,0.888885,0.933100,0.924113,0.917214,0.917975,0.925331,0.933993,0.896743,0.900013,...,0.914674,0.944056,0.929231,0.878757,0.925007,0.931545,0.913668,0.885302,0.889841,0.927906
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486422,rs9363764,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
486423,rs939290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
486424,rs951295,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
486425,rs966367,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Dimensions of the DataFrame:
(486427, 521)

Displaying the DataFrame for BRCA:


,Composite Element REF,a5ba0826-cb02-4247-815c-de4ea8c56180,826fe327-9f91-4f10-913f-07b1d4bd0e52,ff83ac32-c072-42dc-bd53-7e0b13223f36,b2c405d0-0bea-449e-80be-b6779c1540d3,7a4a5192-3f01-442e-ae1a-015eff6fb8ac,cb30aa43-12b5-40db-a66d-d0584e463fe2,57112f31-12e1-44b7-87e3-0cb90e30452b,85e3cec3-d608-44f3-a636-6e5311c28184,b670185e-fd9f-4308-9b83-7dd3cd830cdf,...,35ada957-4511-4f25-a9a7-50a6e51e9fd8,6e37de2e-dab5-4182-9bf1-23ae4230bc3d,9eec9526-ab02-4c55-ba15-97454c1f3880,a69bf894-85dc-4b5c-a509-151c2480727c,77fe848b-5096-4049-88e1-3c05fa785c53,890c9884-52ce-4a12-ac1a-6830c028c66d,bb8f963a-36e6-4b41-900c-f43c5c638dec,532ad62b-576b-42dd-99c7-94d07a1676e4,8895bb33-df02-4203-9b86-72ea61aa9f21,05435bfd-dfdb-4635-9638-ead8fb7d64df
0,cg00000292,0.397227,0.182887,0.581844,NaN,0.588400,0.859486,0.712158,0.402791,0.109326,...,0.822416,0.196123,0.187973,0.115818,0.202879,NaN,0.537524,0.116270,0.089365,0.919856
1,cg00002426,0.459100,0.971166,0.659344,0.949928,0.127865,0.723370,0.028575,0.183314,0.967111,...,0.352582,0.954857,0.960751,0.962906,0.965888,0.973401,0.038170,0.943535,0.951152,0.544520
2,cg00003994,0.026539,0.934178,0.030101,0.915857,0.165752,0.108289,0.036621,0.718537,0.874484,...,0.056386,0.922638,0.853199,0.909111,0.860280,0.915200,0.419862,0.784011,0.900171,0.071922
3,cg00005847,0.584244,0.473353,0.358744,0.306333,0.805335,0.302415,0.352456,0.781428,0.171641,...,0.260832,NaN,0.418528,0.226500,0.131820,0.560529,0.561716,0.216845,0.640216,0.599678
4,cg00006414,0.042373,0.906086,0.039898,0.943385,0.037270,0.055591,0.041465,0.044889,0.914312,...,0.040399,0.888593,0.932323,0.943614,0.944434,0.924803,0.039292,0.882377,0.938991,0.087625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
486422,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
486423,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
486424,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
486425,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Dimensions of the DataFrame:
(486427, 1058)


## Dimensionality Reduction

For the preprocessing of DNA methylation data, three scenarios were studied. The first scenario, based on variance computation, involved calculating the variance across patients for each gene (Composite Element), determining a percentile threshold to filter genes with higher variability (following the Multisurv approach), removing genes with variance above this threshold, imputing missing values using the mean of each row, and finally computing the dimensionality reduction percentage. The second scenario applied Principal Component Analysis (PCA), considering three configurations: selecting the top 100 genes, 150 genes and another selecting the top 200 genes based on their contribution to total variance.

In SAMVAE, we present the results based on a selected set of 100 genes. We chose to reduce the feature space to 100 principal components using PCA because it is a commonly used dimensionality in the literature, aligning with the state-of-the-art approaches in omics data integration for cancer analysis.



### 1. Compute variance 

1) Calculate the variance across columns (patients) for each row (Composite Element),
2) Calculate the percentile for filtering genes with higher variance (as in Multisurv)
3) Filter genes with variance greater than the calculated percentile
4) Impute missing values with the mean of each row
5) Calculate the dimensionality reduction percentage

In [4]:
# Base path where the .pkl files are stored
base_path = "../data_download/pkl"
input_pickle_template = os.path.join(base_path, 'adn_{}.pkl')

# Output paths for the processed methylation data
processed_output_paths = {
    'BRCA': '../samvae-main/data_preprocessing/raw_data/omic_data/adn/brca/processed_adn_BRCA.pkl',
    'LGG': '../samvae-main/data_preprocessing/raw_data/omic_data/adn/lgg/processed_adn_LGG.pkl'
}

# Function to process methylation data
def process_methylation_data(input_pickle_path, output_pickle_path, max_genes=500):
    # Load the original methylation data
    methylation_df = pd.read_pickle(input_pickle_path)
    print(f"Original DataFrame dimensions ({input_pickle_path}): {methylation_df.shape}")

    # Compute variance across patients for each gene
    variance = methylation_df.iloc[:, 2:].var(axis=1, ddof=0)
    gene_variance_df = pd.DataFrame({
        'Composite Element REF': methylation_df['Composite Element REF'],
        'Variance': variance
    })

    # Compute the variance threshold to keep only top `max_genes`
    threshold = gene_variance_df['Variance'].quantile(1 - (max_genes / gene_variance_df.shape[0]))
    print(f"Variance threshold to retain up to {max_genes} genes: {threshold}")

    # Filter genes with variance above the threshold
    selected_genes = gene_variance_df[gene_variance_df['Variance'] > threshold]
    print(f"Number of selected genes: {selected_genes.shape[0]}")

    # Filter the original dataframe
    selected_gene_ids = selected_genes['Composite Element REF'].tolist()
    filtered_df = methylation_df[methylation_df['Composite Element REF'].isin(selected_gene_ids)].copy()

    # Impute missing values with the mean of each row (gene)
    filtered_df.loc[:, filtered_df.columns[1:]] = filtered_df.iloc[:, 1:].apply(
        lambda row: row.fillna(row.mean()), axis=1
    )

    # Transpose the DataFrame (genes as columns, patients as rows)
    transposed_df = filtered_df.set_index('Composite Element REF').T
    transposed_df.to_pickle(output_pickle_path)
    print(f"Processed data saved to: {output_pickle_path}")


# Cancer types to process
cancer_types = ['BRCA', 'LGG']
cancer_to_show = 'LGG'  # Set which cancer type's head you want to display

# Process each cancer type
for cancer_type in cancer_types:
    input_pickle_path = input_pickle_template.format(cancer_type)
    output_pickle_path = processed_output_paths[cancer_type]

    if os.path.exists(input_pickle_path):
        processed_df = process_methylation_data(input_pickle_path, output_pickle_path)
 
    else:
        print(f"Pickle file not found for {cancer_type}: {input_pickle_path}")


Original DataFrame dimensions (../data_download/pkl/adn_BRCA.pkl): (486427, 1058)
Variance threshold to retain up to 500 genes: 0.17162806269213504
Number of selected genes: 437
Processed data saved to: ../samvae-main/data_preprocessing/raw_data/omic_data/adn/brca/processed_adn_BRCA.pkl
Original DataFrame dimensions (../data_download/pkl/adn_LGG.pkl): (486427, 521)
Variance threshold to retain up to 500 genes: 0.09896647792719056
Number of selected genes: 434
Processed data saved to: ../samvae-main/data_preprocessing/raw_data/omic_data/adn/lgg/processed_adn_LGG.pkl


### 2. Principal Components Analysis (PCA)

In [7]:
warnings.filterwarnings('ignore')

# Input path where raw pickle files are stored
input_path = "../data_download/pkl"
input_pickle_template = os.path.join(input_path, 'adn_{}.pkl')

# Output paths for PCA-processed data
output_pca_paths = {
    "LGG": "../samvae-main/data_preprocessing/pca_data/omic_data/adn/lgg/",
    "BRCA": "../samvae-main/data_preprocessing/pca_data/omic_data/adn/brca/"
}

# List of PCA component counts to generate
components_list = [100, 150, 200]

# Function to apply PCA and save results
def apply_pca_and_save(df, cancer_type, n_components, output_dir):
    # Ensure sufficient features for PCA
    if df.shape[1] < n_components:
        print(f"[Warning] Not enough features to reduce to {n_components} components for {cancer_type}. Using {df.shape[1]} components instead.")
        n_components = df.shape[1]

    # Apply PCA
    pca = PCA(n_components=n_components)
    transformed = pca.fit_transform(df)

    # Convert to DataFrame
    df_pca = pd.DataFrame(transformed, index=df.index, columns=[f'PC{i+1}' for i in range(n_components)])

    # Normalize the PCA components
    scaler = MinMaxScaler()
    df_normalized = pd.DataFrame(scaler.fit_transform(df_pca), index=df.index, columns=df_pca.columns)

    # Save to .pkl file
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, f'adn_{cancer_type}_pca_{n_components}.pkl')
    df_normalized.to_pickle(output_file)
    print(f"PCA file ({n_components} components) saved to: {output_file}\n")
    
for cancer_type in ["LGG", "BRCA"]:
    input_file = input_pickle_template.format(cancer_type)

    if os.path.exists(input_file):
        print(f"Processing PCA for: {input_file}")

        # Load data
        df = pd.read_pickle(input_file)

        # Set first column as index  
        df = df.set_index('Composite Element REF')

        # Ensure numeric values, impute missing values with column mean
        df = df.apply(pd.to_numeric, errors='coerce').fillna(df.mean())

        # Transpose to shape: patients x features
        df_numeric = df.T

        # Apply PCA for each number of components
        for n_components in components_list:
            apply_pca_and_save(df_numeric, cancer_type, n_components, output_pca_paths[cancer_type])

    else:
        print(f"[Error] File not found: {input_file}\n")

print("PCA processing completed for all cancer types.")

Processing PCA for: ../data_download/pkl/adn_LGG.pkl
PCA file (100 components) saved to: ../samvae-main/data_preprocessing/pca_data/omic_data/adn/lgg/adn_LGG_pca_100.pkl

PCA file (150 components) saved to: ../samvae-main/data_preprocessing/pca_data/omic_data/adn/lgg/adn_LGG_pca_150.pkl

PCA file (200 components) saved to: ../samvae-main/data_preprocessing/pca_data/omic_data/adn/lgg/adn_LGG_pca_200.pkl

PCA processing completed for all cancer types.


# Copy Number Variation (CNV)

 Copy Number Variation (CNV) involves duplications or deletions of DNA segments and plays a key role in cancer genomics. In this notebook, CNV data were obtained from Affymetrix SNP 6.0 arrays to ensure consistent resolution. We used ABSOLUTE, a tool that accounts for tumor purity and estimates ploidy, to classify segments as **Amplified** (copy number > 2), **Deleted** (< 2), or **Neutral** (= 2).

To focus on biologically relevant regions, we filtered the dataset using a pre-curated list of frequently mutated genes from the GDC, reducing the initial set of 60,624 genes to 711 per cancer type. This approach follows established practices in cancer genomics, such as those used in TCGA studies. To obtain this list, we navigated the GDC portal, filtered the dataset by cancer type (e.g., LGG or BRCA), and selected genes based on Mutation Frequency (see the figure below).


<img src="../data_download/Fig/gdc4.png" alt="Descripción de la imagen" width="1000"/>

This allows us to identify the most frequently mutated genes for each cancer type. For example, in the following figure, the frequently mutated genes for Lower Grade Glioma (LGG) are shown. We downloaded the corresponding TSV file to use this curated gene list in our analysis.


<img src="../data_download/Fig/gdc5.png" alt="Descripción de la imagen" width="1000"/>

In [8]:
def process_cnv_data(cancer_type): 
    # Define paths
    base_path = Path("../data")
    cnv_dir = base_path / cancer_type.upper() / "omics" / "cnv"
    mutated_genes_path = Path("../data_download/Frequently_mutated_genes") / f"frequently-mutated-genes_{cancer_type.lower()}.tsv"

    # Load frequently mutated genes
    mutated_genes = pd.read_csv(mutated_genes_path, sep="\t")
    mutated_genes['gene_id'] = mutated_genes['gene_id'].str.split('.', n=1).str[0]
    mutated_gene_ids = set(mutated_genes['gene_id'])

    # Process each sample folder
    for folder in cnv_dir.iterdir():
        if folder.is_dir():
            for file in folder.glob("*.tsv"):
                # Read CNV data
                cnv_data = pd.read_csv(file, sep="\t")
                cnv_data['gene_id'] = cnv_data['gene_id'].str.split('.', n=1).str[0]

                # Filter for frequently mutated genes
                filtered_data = cnv_data[cnv_data['gene_id'].isin(mutated_gene_ids)]

                # Save filtered file
                filtered_file = folder / f"filtered_{file.name}"
                filtered_data.to_csv(filtered_file, sep="\t", index=False)

# Run the function for LGG and BRCA
process_cnv_data("LGG")
process_cnv_data("BRCA")


In [9]:
# Base directories containing patient folders for LGG and BRCA
base_paths = {
    "LGG": Path("../data/LGG/omics/cnv"),
    "BRCA": Path("../data/BRCA/omics/cnv")
}

# Estimated ploidy for the samples  
ploidy = 2

# Dictionary to store gene classification data per patient and cancer type
data_by_gene_dict = {
    "LGG": {},
    "BRCA": {}
}

# Traverse patient folders for each cancer type
for cancer_type, base_path in base_paths.items():
    for patient_folder in base_path.iterdir():
        if patient_folder.is_dir():
            # Search for files starting with "filtered"
            for file_path in patient_folder.glob("filtered*.tsv"):
                try:
                    df = pd.read_csv(file_path, sep="\t")

                    # Classify segments based on copy number using ABSOLUTE logic
                    df["Classification"] = df["copy_number"].apply(
                        lambda cn: "Amplified" if cn > ploidy else 
                                   "Deleted" if cn < ploidy else "Neutral"
                    )

                    # Organize data by gene and patient
                    for _, row in df.iterrows():
                        gene = row["gene_id"]
                        classification = row["Classification"]

                        if gene not in data_by_gene_dict[cancer_type]:
                            data_by_gene_dict[cancer_type][gene] = {}

                        data_by_gene_dict[cancer_type][gene][patient_folder.name] = classification

                except Exception as e:
                    print(f"Error processing file {file_path}: {e}")

# Save tables per cancer type
for cancer_type, data_by_gene in data_by_gene_dict.items():
    final_df = pd.DataFrame.from_dict(data_by_gene, orient="index")
    final_df.index.name = "Gene"

    output_dir = Path(f"../samvae-main/data_preprocessing/raw_data/omic_data/cnv/{cancer_type.lower()}")
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "consolidated_segments_by_gene.tsv"

    final_df.to_csv(output_file, sep="\t")
    print(f"Consolidated table saved at: {output_file}")


Consolidated table saved at: ../samvae-main/data_preprocessing/raw_data/omic_data/cnv/lgg/consolidated_segments_by_gene.tsv
Consolidated table saved at: ../samvae-main/data_preprocessing/raw_data/omic_data/cnv/brca/consolidated_segments_by_gene.tsv


## PCA

In [10]:
warnings.filterwarnings('ignore')

# Input and output paths
input_paths = {
    "LGG": Path("../samvae-main/data_preprocessing/raw_data/omic_data/cnv/lgg/consolidated_segments_by_gene.tsv"),
    "BRCA": Path("../samvae-main/data_preprocessing/raw_data/omic_data/cnv/brca/consolidated_segments_by_gene.tsv")
}

output_paths = {
    "LGG": Path("../samvae-main/data_preprocessing/pca_data/omic_data/cnv/lgg/"),
    "BRCA": Path("../samvae-main/data_preprocessing/pca_data/omic_data/cnv/brca/")
}

# PCA components
components_list = [100, 150, 200]

# Loop over each cancer type
for cancer_type, input_file in input_paths.items():
    if not input_file.exists():
        print(f"File not found: {input_file}")
        continue

    print(f"Processing PCA for {cancer_type}...")

    # Load CNV data
    df = pd.read_csv(input_file, sep="\t", index_col=0)

    # Load manifest and reorder columns
    manifest_file = Path(f"../data_download/Manifest/{cancer_type}/filtered_manifest/filtered_gdc_manifest.cnv.txt")
    manifest_df = pd.read_csv(manifest_file, sep="\t")
    manifest_ids = manifest_df["id"].tolist()
    df = df[[col for col in manifest_ids if col in df.columns]]

    # Map categorical values
    df.replace({"Neutral": 0, "Amplified": 1, "Deleted": 2}, inplace=True)

    # Ensure numeric types and fill missing values
    df = df.apply(pd.to_numeric, errors='coerce').fillna(df.mean())

    # Transpose: rows = samples, columns = genes
    df_T = df.T

    for n_components in components_list:
        used_components = min(n_components, df_T.shape[1])
        if used_components < n_components:
            print(f"Warning: only {used_components} features available for {n_components} components in {cancer_type}.")

        # Apply PCA
        pca = PCA(n_components=used_components)
        reduced = pca.fit_transform(df_T)

        # Normalize result
        df_pca = pd.DataFrame(reduced, index=df_T.index, columns=[f"PC{i+1}" for i in range(used_components)])
        df_pca_scaled = pd.DataFrame(MinMaxScaler().fit_transform(df_pca), index=df_pca.index, columns=df_pca.columns)

        # Create output directory
        output_dir = output_paths[cancer_type]
        output_dir.mkdir(parents=True, exist_ok=True)
        output_file = output_dir / f"cnv_{cancer_type}_pca_{used_components}.pkl"
        df_pca_scaled.to_pickle(output_file)

        print(f"PCA with {used_components} components saved to: {output_file}\n")

print("PCA processing completed.")


Processing PCA for LGG...
PCA with 100 components saved to: ../samvae-main/data_preprocessing/pca_data/omic_data/cnv/lgg/cnv_LGG_pca_100.pkl

PCA with 150 components saved to: ../samvae-main/data_preprocessing/pca_data/omic_data/cnv/lgg/cnv_LGG_pca_150.pkl

PCA with 200 components saved to: ../samvae-main/data_preprocessing/pca_data/omic_data/cnv/lgg/cnv_LGG_pca_200.pkl

Processing PCA for BRCA...
PCA with 100 components saved to: ../samvae-main/data_preprocessing/pca_data/omic_data/cnv/brca/cnv_BRCA_pca_100.pkl

PCA with 150 components saved to: ../samvae-main/data_preprocessing/pca_data/omic_data/cnv/brca/cnv_BRCA_pca_150.pkl

PCA with 200 components saved to: ../samvae-main/data_preprocessing/pca_data/omic_data/cnv/brca/cnv_BRCA_pca_200.pkl

PCA processing completed.


# microRNA (miRNA)

miRNAs are small non-coding RNAs (20–24 nt) involved in post-transcriptional gene regulation. Their expression levels, quantified as reads per million (RPM), can serve as biomarkers in diseases like cancer. To retain all potential biological signals, the dataset includes all 1,881 miRNAs—regardless of expression level—with no variance-based filtering. Expression values were normalized between 0 and 1.

In [11]:
# Define base and output paths
base_path = "../data" 
output_pickle_template = os.path.join(base_path, 'miRNA_{}.pkl')

# Define manifest paths
manifest_paths = {
    'BRCA': '../data_download/Manifest/BRCA/filtered_manifest/filtered_gdc_manifest.miRNA.txt',
    'LGG': '../data_download/Manifest/LGG/filtered_manifest/filtered_gdc_manifest.miRNA.txt',
}

# Process each cancer type
for cancer_type in ['BRCA', 'LGG']:
    cancer_dir = os.path.join(base_path, cancer_type, 'omics', 'miRNA')

    if not os.path.isdir(cancer_dir):
        print(f"Directory not found: {cancer_dir}")
        continue

    # Load manifest IDs in order
    manifest_df = pd.read_csv(manifest_paths[cancer_type], sep='\t')
    manifest_ids = manifest_df['id'].tolist()

    miRNA_ID = None
    data_by_patient = {}

    for patient_id in manifest_ids:
        patient_dir = os.path.join(cancer_dir, patient_id)
        if not os.path.isdir(patient_dir):
            continue

        # Find the quantification file
        quant_files = [f for f in os.listdir(patient_dir) if f.endswith('mirbase21.mirnas.quantification.txt')]
        if not quant_files:
            continue

        file_path = os.path.join(patient_dir, quant_files[0])
        df = pd.read_csv(file_path, sep='\t', header=None, names=[
            'miRNA_ID', 'read_count', 'reads_per_million_miRNA_mapped', 'cross-mapped'])

        if miRNA_ID is None:
            miRNA_ID = df['miRNA_ID']

        data_by_patient[patient_id] = df['reads_per_million_miRNA_mapped']

    # Create combined DataFrame
    combined_df = pd.DataFrame({'miRNA_ID': miRNA_ID})
    for patient_id in manifest_ids:
        combined_df[patient_id] = data_by_patient.get(patient_id, [None] * len(miRNA_ID))

    # Save combined data to pickle
    output_file = output_pickle_template.format(cancer_type)
    combined_df.to_pickle(output_file)
    print(f"Combined data saved to: {output_file}")


Combined data saved to: ../data/miRNA_BRCA.pkl
Combined data saved to: ../data/miRNA_LGG.pkl


In [12]:
warnings.filterwarnings('ignore')

# Base paths
base_path = "../data"
output_paths = {
    'BRCA': '../samvae-main/data_preprocessing/raw_data/omic_data/miRNA/brca/',
    'LGG': '../samvae-main/data_preprocessing/raw_data/omic_data/miRNA/lgg/'
}

# Process each cancer type
for cancer_type in ['BRCA', 'LGG']:
    input_file = os.path.join(base_path, f'miRNA_{cancer_type}.pkl')
    
    if not os.path.exists(input_file):
        print(f'File not found: {input_file}\n')
        continue

    print(f'Processing {cancer_type}')
    
    # Load pickle file
    df = pd.read_pickle(input_file)
    
    # Separate the first row (miRNA_ID)
    id_row = df.iloc[[0]]
    data = df.iloc[1:].reset_index(drop=True)

    # Scale patient expression data
    scaler = MinMaxScaler()
    data[data.columns[1:]] = scaler.fit_transform(data.iloc[:, 1:])

    # Recombine miRNA_ID row with scaled data
    df_scaled = pd.concat([id_row, data], ignore_index=True)

    # Ensure output directory exists
    os.makedirs(output_paths[cancer_type], exist_ok=True)

    # Save the scaled DataFrame
    output_file = os.path.join(output_paths[cancer_type], f'miRNA_{cancer_type}_scaled.pkl')
    df_scaled.to_pickle(output_file)

    print(f'Scaled data saved to: {output_file}\nShape: {df_scaled.shape}\n')

print("Processing complete for all cancer types.")


Processing BRCA
Scaled data saved to: ../samvae-main/data_preprocessing/raw_data/omic_data/miRNA/brca/miRNA_BRCA_scaled.pkl
Shape: (1882, 1058)

Processing LGG
Scaled data saved to: ../samvae-main/data_preprocessing/raw_data/omic_data/miRNA/lgg/miRNA_LGG_scaled.pkl
Shape: (1882, 521)

Processing complete for all cancer types.


## PCA

In [13]:
warnings.filterwarnings('ignore')

# Input and output paths
input_path = "../data"
input_file_template = os.path.join(input_path, 'miRNA_{}.pkl')

output_paths = {
    'BRCA': '../samvae-main/data_preprocessing/pca_data/omic_data/miRNA/brca/',
    'LGG': '../samvae-main/data_preprocessing/pca_data/omic_data/miRNA/lgg/'
}

# Cancer types and PCA dimensions to test
cancer_types = ['BRCA', 'LGG']
pca_components = [100, 150, 200]

# Process each cancer type
for cancer in cancer_types:
    input_file = input_file_template.format(cancer)
    
    if not os.path.exists(input_file):
        print(f'File not found: {input_file}\n')
        continue

    print(f'Running PCA for {cancer}')
    
    # Load data
    df = pd.read_pickle(input_file)
    
    # Skip the first row and first column (e.g., miRNA_ID), keep numeric patient data
    data = df.iloc[1:, 1:]
    data = data.apply(pd.to_numeric, errors='coerce')

    # Transpose and fill NaNs with column mean (now rows = patients, columns = features)
    data = data.T.fillna(data.mean()).copy()

    for n in pca_components:
        n_actual = min(n, data.shape[1])
        if n_actual < n:
            print(f'Warning: Not enough features to reduce to {n} components in {cancer}. Using {n_actual} instead.')

        # Perform PCA
        pca = PCA(n_components=n_actual)
        pca_result = pca.fit_transform(data)

        # Convert to DataFrame
        df_pca = pd.DataFrame(pca_result, columns=[f'PC{i+1}' for i in range(n_actual)])

        # Optionally normalize PCA output
        scaler = MinMaxScaler()
        df_pca_scaled = pd.DataFrame(scaler.fit_transform(df_pca), columns=df_pca.columns)

        # Output information
        print(f'PCA result shape ({cancer}, {n_actual} components): {df_pca_scaled.shape}')

        # Create output directory if needed
        os.makedirs(output_paths[cancer], exist_ok=True)

        # Save PCA-transformed data
        output_file = os.path.join(output_paths[cancer], f'miRNA_{cancer}_pca_{n_actual}.pkl')
        df_pca_scaled.to_pickle(output_file) 

Running PCA for BRCA
PCA result shape (BRCA, 100 components): (1057, 100)
PCA result shape (BRCA, 150 components): (1057, 150)
PCA result shape (BRCA, 200 components): (1057, 200)
Running PCA for LGG
PCA result shape (LGG, 100 components): (520, 100)
PCA result shape (LGG, 150 components): (520, 150)
PCA result shape (LGG, 200 components): (520, 200)


# RNA-sequencing (RNAseq)

RNA-seq is a technique used to quantify and analyze the transcriptome—the complete set of RNA molecules expressed in a biological sample. In this analysis, reads were aligned using the GENCODE v36 annotation, and gene expression was quantified using normalized metrics such as FPKM and FPKM-UQ, which account for gene length and sequencing depth. To enable cross-sample comparisons, reads are treated as unstranded, meaning no orientation is assigned to RNA strands. As a preprocessing step, the 1,000 most variable genes (by expression variance) are selected to reduce dimensionality, and additional feature selection techniques such as the mRMR criterion—previously used in DNA methylation analysis—are also applied.



In [14]:
# Define base and output paths
base_path = "../data"
output_path = base_path
output_pickle_template = os.path.join(output_path, 'RNAseq_{}.pkl')

# Define manifest paths for each cancer type
manifest_paths = {
    'BRCA': '../data_download/Manifest/BRCA/filtered_manifest/filtered_gdc_manifest.RNAseq.txt',
    'LGG': '../data_download/Manifest/LGG/filtered_manifest/filtered_gdc_manifest.RNAseq.txt',
}

# Process RNAseq data for each cancer type
for cancer_type in ['BRCA', 'LGG']:
    rna_seq_dir = os.path.join(base_path, cancer_type, 'omics', 'RNAseq')
    if not os.path.isdir(rna_seq_dir):
        print(f"[Skipped] RNAseq folder not found for {cancer_type}: {rna_seq_dir}")
        continue

    # Load manifest to get patient IDs
    manifest_file = manifest_paths[cancer_type]
    manifest_df = pd.read_csv(manifest_file, sep='\t')
    patient_ids = manifest_df['id'].tolist()

    gene_ids = None
    expression_data = {}

    for patient_id in patient_ids:
        patient_dir = os.path.join(rna_seq_dir, patient_id)
        if not os.path.isdir(patient_dir):
            continue

        # Search for the RNAseq file
        file_found = False
        for fname in os.listdir(patient_dir):
            if fname.endswith('rna_seq.augmented_star_gene_counts.tsv'):
                file_path = os.path.join(patient_dir, fname)
                df = pd.read_csv(file_path, sep='\t', comment='#')

                # Filter valid genes and extract FPKM-UQ values
                valid_df = df[df['gene_id'].str.startswith('ENSG')][['gene_id', 'fpkm_uq_unstranded']]

                if not valid_df.empty:
                    if gene_ids is None:
                        gene_ids = valid_df['gene_id'].reset_index(drop=True)
                    expression_data[patient_id] = valid_df['fpkm_uq_unstranded'].reset_index(drop=True)
                    file_found = True
                break

        if not file_found:
            print(f"[Warning] No RNAseq file found for patient {patient_id} in {cancer_type}.")

    if gene_ids is None or not expression_data:
        print(f"[Error] No valid RNAseq data found for {cancer_type}.")
        continue

    # Create list of Series for each patient (to avoid fragmentation)
    data_columns = []
    for pid in patient_ids:
        values = expression_data.get(pid, [None] * len(gene_ids))
        data_columns.append(pd.Series(values, name=pid))

    # Concatenate all patient columns into a single DataFrame
    expression_df = pd.concat(data_columns, axis=1)

    # Insert gene IDs as the first column
    expression_df.insert(0, 'gene_id', gene_ids)

    # Save the combined DataFrame to a pickle file
    output_file = output_pickle_template.format(cancer_type)
    expression_df.to_pickle(output_file)
    print(f"Combined RNAseq data saved to: {output_file}")


Combined RNAseq data saved to: ../data/RNAseq_BRCA.pkl
Combined RNAseq data saved to: ../data/RNAseq_LGG.pkl


## Dimensionality Reduction

### Variance

In [15]:
# Define cancer types to process
cancer_types = ['BRCA', 'LGG']

# Template for input pickle files
input_pickle_template = "../data/RNAseq{}.pkl"

# Output paths for filtered data
filtered_output_paths = {
    'BRCA': '../samvae-main/data_preprocessing/raw_data/omic_data/RNAseq/brca/',
    'LGG': '../samvae-main/data_preprocessing/raw_data/omic_data/RNAseq/lgg/'
}

# Number of most variable genes to retain
top_genes_count = 1000

for cancer_type in cancer_types:
    input_pickle_file = input_pickle_template.format(cancer_type)
    
    if not os.path.isfile(input_pickle_file):
        print(f"[Skipped] File not found for {cancer_type}: {input_pickle_file}")
        continue

    # Load RNAseq data
    df = pd.read_pickle(input_pickle_file)

    # Compute variance per gene (across patients)
    variance_series = df.set_index('gene_id').T.var(axis=0)

    # Determine cutoff percentile for top variable genes
    quantile_threshold = (variance_series.shape[0] - top_genes_count) / variance_series.shape[0]
    print(f"\n{cancer_type}: Retaining top {top_genes_count} variable genes "
          f"(above {round(quantile_threshold * 100, 1)} percentile)")

    # Filter top variable genes
    top_genes = variance_series[variance_series > variance_series.quantile(quantile_threshold)].index
    print(f"{cancer_type}: Selected {len(top_genes)} genes")

    # Filter original DataFrame
    filtered_df = df[df['gene_id'].isin(top_genes)].reset_index(drop=True)

    # Create output directory if it doesn't exist
    output_dir = filtered_output_paths[cancer_type]
    os.makedirs(output_dir, exist_ok=True)

    # Save filtered DataFrame
    output_pickle_file = os.path.join(output_dir, f'RNAseq{cancer_type}_filtered.pkl')
    filtered_df.to_pickle(output_pickle_file)



BRCA: Retaining top 1000 variable genes (above 98.4 percentile)
BRCA: Selected 1000 genes

LGG: Retaining top 1000 variable genes (above 98.4 percentile)
LGG: Selected 1000 genes


### PCA

In [16]:
warnings.filterwarnings('ignore')

# Input and output paths
input_paths = {
    "LGG": "../samvae-main/data_preprocessing/raw_data/omic_data/RNAseq/lgg/RNAseqLGG_filtered.pkl",
    "BRCA": "../samvae-main/data_preprocessing/raw_data/omic_data/RNAseq/brca/RNAseqBRCA_filtered.pkl"
}

output_pca_paths = {
    "LGG": "../samvae-main/data_preprocessing/pca_data/omic_data/RNAseq/lgg/",
    "BRCA": "../samvae-main/data_preprocessing/pca_data/omic_data/RNAseq/brca/"
}

# List of PCA components to reduce to
components_list = [100, 150, 200]

# Process each cancer type
for cancer_type, input_file in input_paths.items():
    if not os.path.exists(input_file):
        print(f"File not found: {input_file}\n")
        continue

    print(f"PCA for: {input_file}")

    # Load the filtered RNAseq data
    df = pd.read_pickle(input_file)

    # Set 'gene_id' as index
    df = df.set_index('gene_id')

    # Convert all data to numeric and impute missing values with column mean
    df = df.apply(pd.to_numeric, errors='coerce').fillna(df.mean())

    # Transpose so samples are rows
    df_transposed = df.T

    for n_components in components_list:
        n_features = df_transposed.shape[1]
        actual_components = min(n_components, n_features)

        if actual_components < n_components:
            print(f"[Warning] Not enough features to reduce to {n_components} components for {cancer_type}. Using {actual_components} instead.")

        # Apply PCA
        pca = PCA(n_components=actual_components)
        transformed = pca.fit_transform(df_transposed)

        # Convert to DataFrame and normalize
        df_pca = pd.DataFrame(transformed, index=df_transposed.index, 
                              columns=[f'PC{i+1}' for i in range(actual_components)])
        df_pca_normalized = pd.DataFrame(MinMaxScaler().fit_transform(df_pca), 
                                         index=df_pca.index, columns=df_pca.columns)

        # Ensure output directory exists
        os.makedirs(output_pca_paths[cancer_type], exist_ok=True)

        # Save to pickle file
        output_file = os.path.join(output_pca_paths[cancer_type], f'RNAseq_{cancer_type}_pca_{actual_components}.pkl')
        df_pca_normalized.to_pickle(output_file)

PCA for: ../samvae-main/data_preprocessing/raw_data/omic_data/RNAseq/lgg/RNAseqLGG_filtered.pkl
PCA for: ../samvae-main/data_preprocessing/raw_data/omic_data/RNAseq/brca/RNAseqBRCA_filtered.pkl
